# Gathering HLS4ML-reports


In [23]:
import hls4ml 

search_dir = '../'

patterns = { 
    "All HLS4ML-projects"  : "hls4ml_prj*/",
    #"MNIST HLS4ML-projects" : "MNIST/*/hls4ml_prj*/",  
    #"SVHN HLS4ML-projects"  : "SVHN/*/hls4ml_prj*/",  
}

save_reports = 'reports/'

In [24]:
import os
from pathlib import Path
import sys

#os.listdir('../', )
reports = {}

for label,pattern in patterns.items():
    print(f"Searching {label} ({pattern})")
    for path in Path(search_dir).rglob(pattern):
        if path.is_dir():
            # https://docs.python.org/3/library/pathlib.html#pathlib.PurePath
            #path = path.absolute()
            project_name = f"{path.parts[-3]}-{path.parts[-2]}-{path.name}" # Dataset - Modellarkitektur - hva HLS4ML-prosjektet heter
            print(f"\n\n%%%%%%% {project_name} found in {path} %%%%%%%%%%%\n")
            report_path = f"{save_reports}/{project_name}.rpt"
            with open(report_path, 'w') as f:
                sys.stdout = f
                hls4ml.report.read_vivado_report(str(path), full_report=True) # https://github.com/fastmachinelearning/hls4ml/blob/main/hls4ml/report/vivado_report.py
                sys.stdout = sys.__stdout__
                # parse report makes a "summary"
            summary = hls4ml.report.parse_vivado_report(str(path))
            reports.update({
                project_name : {
                    'path' : str(path.absolute()),
                    'summary' : summary,
                    'report_path' : str(report_path)
                    }
                })

In [25]:
display(reports)

{'SVHN-Qkeras-hls4ml_prj_UniformPrecision': {'path': '/home/ncgadmin/DAT255/DAT255-project/Reports/../SVHN/Qkeras/hls4ml_prj_UniformPrecision',
  'summary': {},
  'report_path': 'reports//SVHN-Qkeras-hls4ml_prj_UniformPrecision.rpt'},
 'MNIST-CNN_HGQ_StaticTraining-hls4ml_prj_MNIST_MLP_HGQ_StaticTraining_model': {'path': '/home/ncgadmin/DAT255/DAT255-project/Reports/../MNIST/CNN_HGQ_StaticTraining/hls4ml_prj_MNIST_MLP_HGQ_StaticTraining_model',
  'summary': {},
  'report_path': 'reports//MNIST-CNN_HGQ_StaticTraining-hls4ml_prj_MNIST_MLP_HGQ_StaticTraining_model.rpt'},
 'MNIST-CNN_HGQ_StaticTraining-hls4ml_prj_{model_name}': {'path': '/home/ncgadmin/DAT255/DAT255-project/Reports/../MNIST/CNN_HGQ_StaticTraining/hls4ml_prj_{model_name}',
  'summary': {},
  'report_path': 'reports//MNIST-CNN_HGQ_StaticTraining-hls4ml_prj_{model_name}.rpt'},
 'MNIST-MLP_HGQ_StaticTraining-hls4ml_prj_MNIST_MLP_HGQ_StaticTraining_model': {'path': '/home/ncgadmin/DAT255/DAT255-project/Reports/../MNIST/MLP_HGQ_

Extract summary to table

In [ ]:
import pandas as pd

summary_fields = [#'CSynthesisReport.BestLatency', 
                  'CSynthesisReport.LUT', 'CSynthesisReport.FF', 'CSynthesisReport.DSP', 'CSynthesisReport.BRAM_18K']

base_columns = ["project_name"]
summary_rows = [
    {
        "project_name": project_name,
        **report["summary"],
    }
    for project_name, report in reports.items()
]

results_table = pd.json_normalize(summary_rows, sep=".")
summary_columns = [field for field in summary_fields if field in results_table.columns]

resource_columns = {
    "CSynthesisReport.LUT": ["AvailableLUT", "CSynthesisReport.AvailableLUT"],
    "CSynthesisReport.FF": ["AvailableFF", "CSynthesisReport.AvailableFF"],
    "CSynthesisReport.DSP": ["AvailableDSP", "CSynthesisReport.AvailableDSP"],
    "CSynthesisReport.BRAM_18K": ["AvailableBRAM_18K", "CSynthesisReport.AvailableBRAM_18K"],
    #"CSynthesisReport.URAM": ["AvailableURAM", "CSynthesisReport.AvailableURAM"],
}

percentage_columns = []
for used_column, available_candidates in resource_columns.items():
    if used_column not in results_table.columns:
        continue

    available_column = next((column for column in available_candidates if column in results_table.columns), None)
    if available_column is None:
        continue

    used = pd.to_numeric(results_table[used_column], errors="coerce")
    available = pd.to_numeric(results_table[available_column], errors="coerce")
    percentage_column = f"{used_column}.pct"
    results_table[percentage_column] = (used / available) * 100
    percentage_columns.append(percentage_column)

results_table = results_table.loc[:, [*base_columns, *summary_columns, *percentage_columns]]
display(results_table.style.format({column: "{:.0f}%" for column in percentage_columns}))


,project_name,CSynthesisReport.BestLatency,CSynthesisReport.LUT,CSynthesisReport.FF,CSynthesisReport.DSP,CSynthesisReport.BRAM_18K,CSynthesisReport.LUT.pct,CSynthesisReport.FF.pct,CSynthesisReport.DSP.pct,CSynthesisReport.BRAM_18K.pct
0,SVHN-Qkeras-hls4ml_prj_UniformPrecision,nan,nan,nan,nan,nan,nan%,nan%,nan%,nan%
1,MNIST-CNN_HGQ_StaticTraining-hls4ml_prj_MNIST_MLP_HGQ_StaticTraining_model,nan,nan,nan,nan,nan,nan%,nan%,nan%,nan%
2,MNIST-CNN_HGQ_StaticTraining-hls4ml_prj_{model_name},nan,nan,nan,nan,nan,nan%,nan%,nan%,nan%
3,MNIST-MLP_HGQ_StaticTraining-hls4ml_prj_MNIST_MLP_HGQ_StaticTraining_model,7,219809,14443,0,0,188%,6%,0%,0%
4,MNIST-baseline-hls4ml_prj_MNIST_baseline_iostream,2366,563346,185530,12254,259,481%,79%,982%,90%
5,MNIST-baseline-hls4ml_prj_MNIST_baseline_model_ioparallel,nan,nan,nan,nan,nan,nan%,nan%,nan%,nan%
6,MNIST-Qkeras-hls4ml_prj_stream_latency,2367,143665,34445,46,134,123%,15%,4%,47%
7,MNIST-Qkeras-hls4ml_prj_base,2367,143665,34445,46,134,123%,15%,4%,47%
8,MNIST-CNN_HGQ_DynamicTraining-hls4ml_prj_{model_name},nan,nan,nan,nan,nan,nan%,nan%,nan%,nan%
9,MNIST-MLP_HGQ_DynamicTraining-hls4ml_prj_MNIST_MLP_HGQ_DynamicTraining,6,54534,5481,0,0,47%,2%,0%,0%
